In [ ]:
# ---------------------------------------------------------
# 0. CREATE DIRECTORIES
#       a. Create directories
#       b. Initialize Faker components
# 1. DATA.ENV PARAMETERS LOAD
# 2. POSTGRESQL ENGINE
# 3. MAIN FUNCTION
#       a. Load CSV data
#       b. Add identity columns based on gender
#       c. Write to SQL database
# ---------------------------------------------------------

# 0. IMPORT LIBRARIES AND CREATE DIRECTORIES
from pathlib import Path
from datetime import datetime
import os, random, urllib.parse
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

np.seterr(divide="ignore", invalid="ignore")
pd.options.display.max_columns = None
pd.set_option("display.max_rows", 100)

today = datetime.today().strftime("%Y-%m-%d")

# a. Create directories
os.makedirs("./Output", exist_ok=True)
os.makedirs("./Input", exist_ok=True)
os.makedirs("./model_save/model", exist_ok=True)

DATA_ENV_PATH = Path("./Input/data.env")
INPUT_CSV_PATH = Path("./Input/ibm_hr_synthetic_data.csv")

# b. Initialize Faker components
from faker import Faker
Faker.seed(42)
random.seed(42)
np.random.seed(42)

fake_sk = Faker("sk_SK")
fake_us = Faker("en_US")
fakers = [fake_sk, fake_us]



# 1. DATA.ENV PARAMETERS LOAD

def load_env_from_file(env_path: Path = DATA_ENV_PATH) -> None:
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")

    with env_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()


def get_env(key: str, default: str | None = None, required: bool = False) -> str | None:
    value = os.getenv(key, default)
    if required and not value:
        raise RuntimeError(f"Missing required env key: {key}")
    return value


# 2. POSTGRESQL ENGINE
def make_pg_engine(env_path: Path = DATA_ENV_PATH) -> sa.Engine:
    load_env_from_file(env_path)

    pg_host = get_env("PG_HOST", required=True)
    pg_port = get_env("PG_PORT", "5432", required=True)
    pg_db = get_env("PG_DATABASE", required=True)
    pg_user = get_env("PG_USERNAME", required=True)
    pg_pwd = get_env("PG_PASSWORD", required=True)

    user_enc = urllib.parse.quote_plus(pg_user)
    pwd_enc = urllib.parse.quote_plus(pg_pwd)

    engine = sa.create_engine(
        f"postgresql+psycopg://{user_enc}:{pwd_enc}@{pg_host}:{pg_port}/{pg_db}",
        pool_pre_ping=True,
        future=True,
    )
    return engine

def get_pg_schema(env_path: Path = DATA_ENV_PATH) -> str:
    load_env_from_file(env_path)
    return get_env("PG_SCHEMA", "public") or "public"

def ensure_schema_exists(engine: sa.Engine, schema_name: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema_name}"'))

def test_db_connection(engine: sa.Engine) -> None:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))

def generate_identity_for_gender(gender: str) -> pd.Series:
    faker = random.choice(fakers)

    if gender == "Male":
        first = faker.first_name_male()
    elif gender == "Female":
        first = faker.first_name_female()
    else:
        first = faker.first_name()

    if faker == fake_sk:
        if gender == "Male":
            last = faker.last_name_male()
        elif gender == "Female":
            last = faker.last_name_female()
        else:
            last = faker.last_name()
    else:
        last = faker.last_name()

    full = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}@sk_company.com"
    username = f"{first[0].lower()}{last.lower()}"

    return pd.Series(
        [first, last, full, email, username],
        index=["FirstName", "LastName", "FullName", "Email", "Username"]
    )

# 3. MAIN FUNCTION

def main() -> None:
    if not INPUT_CSV_PATH.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV_PATH}")
    # a. Load CSV data
    df = pd.read_csv(INPUT_CSV_PATH)
    print("Original shape:", df.shape)

    # b. Add identity columns based on gender
    df[["FirstName", "LastName", "FullName", "Email", "Username"]] = df["Gender"].apply(generate_identity_for_gender)

    cat_cols = [
                "Attrition",
                "Over18",
                "BusinessTravel",
                "Gender",
                "EducationField",
                "EnvironmentSatisfaction",
                "JobInvolvement",
                "JobLevel",
                "JobRole",
                "JobSatisfaction",
                "PerformanceRating",
                "RelationshipSatisfaction",
                "WorkLifeBalance",
                "MaritalStatus",
                "OverTime",
                "Department",
                "DistanceFromHome",
                "Education",
                "FirstName",
                "LastName",
                "FullName",
                "Email",
                "Username",
                ]

    cat_distributions = {
        col: df[col].value_counts(normalize=True)
        for col in cat_cols
        if col in df.columns
    }

    numeric_cols = df.select_dtypes(include=["int64", "int32"]).columns.tolist()

    n_new = len(df) * 10
    synthetic_rows = []

    for _ in range(n_new):
        row = {}

        for col in cat_cols:
            if col not in cat_distributions:
                continue
            vals = cat_distributions[col].index
            probs = cat_distributions[col].values
            row[col] = np.random.choice(vals, p=probs)

        for col in numeric_cols:
            row[col] = int(np.random.choice(df[col].values))

        gender = row.get("Gender", None)
        faker = random.choice(fakers)

        if gender == "Male":
            first = faker.first_name_male()
        elif gender == "Female":
            first = faker.first_name_female()
        else:
            first = faker.first_name()

        if faker == fake_sk:
            if gender == "Male":
                last = faker.last_name_male()
            elif gender == "Female":
                last = faker.last_name_female()
            else:
                last = faker.last_name()
        else:
            last = faker.last_name()

        full = f"{first} {last}"
        email = f"{first.lower()}.{last.lower()}@sk_company.com"
        username = f"{first[0].lower()}{last.lower()}"

        row["FirstName"] = first
        row["LastName"] = last
        row["FullName"] = full
        row["Email"] = email
        row["Username"] = username

        synthetic_rows.append(row)

    df_new = pd.DataFrame(synthetic_rows)
    df_new = df_new.reindex(columns=df.columns)
    df_extended = pd.concat([df, df_new], ignore_index=True)

    if "EmployeeNumber" in df_extended.columns:
        df_extended["EmployeeNumber"] = range(1, len(df_extended) + 1)

    for c in cat_cols:
        if c in df_extended.columns:
            df_extended[c] = df_extended[c].astype("category")

    print("Extended shape:", df_extended.shape)
    print(df_extended.head(10))
    print(df_extended.info())
    
    for c in cat_cols:
        if c in df_extended.columns:
            df_extended[c] = df_extended[c].astype("string")
    # c. Write to SQL database
    engine = make_pg_engine()
    schema_name = get_pg_schema()
    table_name = "HR_Synth_Data"

    test_db_connection(engine)
    ensure_schema_exists(engine, schema_name)

    df_extended.to_sql(table_name,con=engine, if_exists="append", index=False, schema=schema_name,method="multi", chunksize=1000,)

    print(f"Data successfully written to {schema_name}.{table_name}")


if __name__ == "__main__":
    main()